# Stan Gamma-Poisson Random Restarts

Runs Stan ADVI random restarts and plots constrained lambda moments from Stan variational draws.

In [1]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

REL_DIR = Path("single_MC/gamma_poisson/random_restarts")
STAN_FILE_NAME = "stan_gamma_poisson.stan"


def find_notebook_dir():
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / REL_DIR]
    candidates.extend(parent for parent in cwd.parents)
    candidates.extend(parent / REL_DIR for parent in cwd.parents)
    for candidate in candidates:
        if (candidate / STAN_FILE_NAME).exists():
            return candidate.resolve()
    raise FileNotFoundError(f"Could not find {STAN_FILE_NAME} from {cwd}")


NOTEBOOK_DIR = find_notebook_dir()
os.chdir(NOTEBOOK_DIR)

REPO_ROOT = NOTEBOOK_DIR
while not (REPO_ROOT / "modulars").exists():
    if REPO_ROOT.parent == REPO_ROOT:
        raise FileNotFoundError("Could not locate repo root containing modulars/")
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

STAN_FILE = NOTEBOOK_DIR / STAN_FILE_NAME
print(f"Notebook dir: {NOTEBOOK_DIR}")
print(f"Stan model: {STAN_FILE}")

if Path(sys.prefix).name != "stan3":
    raise RuntimeError(
        f"This notebook must run in the miniconda environment named stan3; "
        f"current sys.prefix is {sys.prefix!r}."
    )

STAN3_PREFIX = Path(sys.prefix).resolve()
STAN3_CMDSTAN = STAN3_PREFIX / "bin" / "cmdstan"
if not STAN3_CMDSTAN.exists():
    raise FileNotFoundError(f"Expected CmdStan at {STAN3_CMDSTAN}")

os.environ["CMDSTAN"] = str(STAN3_CMDSTAN)
from cmdstanpy import cmdstan_path, set_cmdstan_path
set_cmdstan_path(str(STAN3_CMDSTAN))

print(f"Python executable: {sys.executable}")
print(f"CmdStan path: {cmdstan_path()}")


Notebook dir: /Users/madelynandersen/Documents/VB Bakeoff/simpleVI/single_MC/gamma_poisson/random_restarts
Stan model: /Users/madelynandersen/Documents/VB Bakeoff/simpleVI/single_MC/gamma_poisson/random_restarts/stan_gamma_poisson.stan
Python executable: /Users/madelynandersen/miniconda3/envs/stan3/bin/python
CmdStan path: /Users/madelynandersen/miniconda3/envs/stan3/bin/cmdstan


/Users/madelynandersen/miniconda3/envs/stan3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from modulars import poisson, print_model_info
from modulars.utils import load_config, load_best_values, exp_lognormal_moments

config_file = NOTEBOOK_DIR.parent / "poisson_config.json"
config = load_config(config_file)

lambda_like = config["lambda_like"]
alpha_prior = config["alpha_prior"]
beta_prior = config["beta_prior"]
n_samples = int(config["n_samples"])

data = poisson(n_samples, lambda_like).astype(int)
print_model_info(
    "Gamma", "Pois", "Gamma",
    [alpha_prior, beta_prior], [lambda_like],
    data=data, n_samples=n_samples,
)

stan_data = {
    "N": len(data),
    "y": data.astype(int),
    "alpha_prior": alpha_prior,
    "beta_prior": beta_prior,
}
PARAM_COLUMNS = ["lambd"]
LOG_COLUMNS = []
best_mu, best_std = load_best_values(config=config, transform=exp_lognormal_moments, n_samples=50_000, seed=1)
best_mu, best_std


Prior: Gamma([1.5, 1.0])
Likelihood: Pois([2.0])
Data: []


(1.4900450871338062, 1.4624227010315984)

In [3]:
from pathlib import Path

RUN_MODE = os.environ.get("SIMPLEVI_STAN_RUN_MODE", "full")  # "quick" or "full"
RUN_CONFIGS = {
    "quick": {"max_iters": 20, "n_restarts": 1, "track_every": 10, "parallel": False, "max_workers": None, "keep_outputs": True},
    "full": {"max_iters": 300_000, "n_restarts": 20, "track_every": 10, "parallel": True, "max_workers": None, "keep_outputs": False},
}

run_config = RUN_CONFIGS[RUN_MODE]
max_iters = int(os.environ.get("SIMPLEVI_STAN_MAX_ITERS", run_config["max_iters"]))
n_restarts = int(os.environ.get("SIMPLEVI_STAN_N_RESTARTS", run_config["n_restarts"]))
track_every = int(os.environ.get("SIMPLEVI_STAN_TRACK_EVERY", run_config["track_every"]))
parallel = bool(int(os.environ.get("SIMPLEVI_STAN_PARALLEL", int(run_config["parallel"]))))
max_workers_env = os.environ.get("SIMPLEVI_STAN_MAX_WORKERS")
max_workers = int(max_workers_env) if max_workers_env else run_config["max_workers"]
keep_outputs = bool(int(os.environ.get("SIMPLEVI_STAN_KEEP_OUTPUTS", int(run_config["keep_outputs"]))))

results_dir = Path(os.environ.get("SIMPLEVI_STAN_RESULTS_DIR", "stan_cmdstan_output"))
results_dir.mkdir(exist_ok=True, parents=True)

run_config | {"parallel": parallel, "max_workers": max_workers, "results_dir": str(results_dir)}


{'max_iters': 300000,
 'n_restarts': 20,
 'track_every': 10,
 'parallel': True,
 'max_workers': None,
 'keep_outputs': False,
 'results_dir': 'stan_cmdstan_output'}

In [4]:
from modulars.stan_rr_test import run_stan_random_restarts, stan_result_by_scenario, FOUR_WAY_STAN_SCENARIOS

TOL_TARGET_ITERS = int(os.environ.get("SIMPLEVI_STAN_TOL_TARGET_ITERS", 200_000))
five_way_scenarios = [dict(s) for s in FOUR_WAY_STAN_SCENARIOS]
for _scenario in five_way_scenarios:
    if _scenario["name"] == "grad100_tol_rel_obj1e-6":
        _scenario["max_iters_scale"] = TOL_TARGET_ITERS / max_iters
five_way_scenarios.append({
    "name": "grad100_tol_rel_obj1e-6_eta0.01",
    "mc_label": "100_tol1e-6_eta0.01",
    "cmd_args": ("grad_samples=100", "adapt", "engaged=0", "eta=0.01"),
    "tol_rel_obj": 1e-6,
    "max_iters_scale": TOL_TARGET_ITERS / max_iters,
    "track_every_scale": 10.0,
})

five_way_scenarios.append({
    "name": "tol_1e-6_default_rest",
    "mc_label": "tol_1e-6_default_rest",
    "cmd_args": (),
    "tol_rel_obj": 1e-6,
    "max_iters_scale": 75_000 / max_iters,
    "track_every_scale": 150 / track_every,
})
stan_result = run_stan_random_restarts(
    stan_file=STAN_FILE,
    data=stan_data,
    param_columns=PARAM_COLUMNS,
    log_columns=LOG_COLUMNS,
    output_dir=results_dir / "cmdstan_runs",
    max_iters=max_iters,
    n_restarts=n_restarts,
    track_every=track_every,
    seed_offset=0,
    parallel=parallel,
    max_workers=max_workers,
    keep_outputs=keep_outputs,
    refresh=0,
    scenarios=five_way_scenarios,
)
if stan_result["failures"]:
    print(f"Stan failures: {len(stan_result['failures'])}; see {results_dir / 'cmdstan_runs' / 'stan_run_failures.csv'}")


Stan restart/scenario tasks:   0%|          | 0/100 [00:00<?, ?it/s]

Stan restart/scenario tasks:   0%|          | 0/100 [00:00<?, ?it/s]

Stan restart/scenario tasks:   1%|          | 1/100 [02:08<3:32:49, 128.99s/it]

Stan restart/scenario tasks:   2%|▏         | 2/100 [02:19<1:36:27, 59.06s/it] 

Stan restart/scenario tasks:   3%|▎         | 3/100 [02:19<52:21, 32.39s/it]  

...

Stan restart/scenario tasks:  99%|█████████▉| 99/100 [10:13:47<06:57, 417.37s/it]

Stan restart/scenario tasks: 100%|██████████| 100/100 [10:15:43<00:00, 326.89s/it]

Stan restart/scenario tasks: 100%|██████████| 100/100 [10:15:43<00:00, 369.43s/it]

Stan restart/scenario tasks:   0%|          | 0/100 [10:15:43<?, ?it/s]

Stan failures: 14005; see stan_cmdstan_output/cmdstan_runs/stan_run_failures.csv


In [5]:
import pickle

STAN_RESULT_PKL = results_dir / "stan_result.pkl"
with open(STAN_RESULT_PKL, "wb") as f:
    pickle.dump(stan_result, f)
print(f"Saved Stan results to {STAN_RESULT_PKL}")

Saved Stan results to stan_cmdstan_output/stan_result.pkl


In [6]:
import pickle

STAN_RESULT_PKL = results_dir / "stan_result.pkl"
with open(STAN_RESULT_PKL, "rb") as f:
    stan_result = pickle.load(f)
if stan_result["failures"]:
    print(f"Stan failures: {len(stan_result['failures'])}; see {results_dir / 'cmdstan_runs' / 'stan_run_failures.csv'}")
print(f"Loaded Stan results from {STAN_RESULT_PKL}")

Stan failures: 14005; see stan_cmdstan_output/cmdstan_runs/stan_run_failures.csv
Loaded Stan results from stan_cmdstan_output/stan_result.pkl


In [7]:
by_scenario = stan_result_by_scenario(stan_result)

default_means, default_stds, default_iterations = by_scenario["default"]
grad100_means, grad100_stds, grad100_iterations = by_scenario["grad100_default_rest"]
adapt_off_means, adapt_off_stds, adapt_off_iterations = by_scenario["grad100_adapt_off"]
tol_means, tol_stds, tol_iterations = by_scenario["grad100_tol_rel_obj1e-6"]
eta_means, eta_stds, eta_iterations = by_scenario["grad100_tol_rel_obj1e-6_eta0.01"]

import numpy as np
from modulars.plot_rr import plot_a_few_trajectories_1d, plot_mean_band_rrs_1d

# "matplotlib" (default, static, savable via plt.savefig) or "plotly"
# (interactive: zoom/pan/hover, shown inline). Flip this and rerun the
# plotting cell below to switch how every plot in this notebook renders.
PLOT_BACKEND = "plotly"


In [7]:
for name, mc_label in zip(stan_result["scenario_names"], stan_result["scenario_mc_labels"]):
    means, stds, scen_iterations = by_scenario[name]
    plot_mean_band_rrs_1d(
        means, stds, best_mu, best_std,
        scen_iterations, r"$\lambda$", mc_label, label_prefix=f"Stan {name} ", backend=PLOT_BACKEND
    )


def downsample_and_extend(arr, factor, target_len):
    """Downsample along the last (iteration) axis by `factor`, then hold
    the last value flat out to `target_len` points -- a valid projection
    once a run has converged and stopped changing, used here to compare
    the (short) default run against the (long) tolerance-ablation run on
    the latter's own iteration grid.
    """
    downsampled = arr[..., ::factor]
    n_have = downsampled.shape[-1]
    if n_have < target_len:
        pad = np.repeat(downsampled[..., -1:], target_len - n_have, axis=-1)
        downsampled = np.concatenate([downsampled, pad], axis=-1)
    return downsampled


# non-tolerance comparisons: these three scenarios share the default
# scenario's own iteration grid (all three use the same max_iters/track_every).
for name in ("grad100_default_rest", "grad100_adapt_off"):
    means, stds, scen_iterations = by_scenario[name]
    plot_a_few_trajectories_1d(
        [default_means, default_stds], [means, stds],
        best_mu, best_std, r"$\lambda$", label_prefix=f"Stan {name} ", x=default_iterations,
        backend=PLOT_BACKEND,
    )

# tolerance comparison: the tolerance scenario runs 2x longer (and is tracked
# 10x coarser) than the default run, so we downsample the default run's
# trajectory by 10x to match the tolerance scenario's tracking step, then
# project its last (converged) value flat out to match tol_iterations'
# length -- it does not change once converged, so this is a valid stand-in
# for actually rerunning the default scenario over the longer window.
projected_default_means = downsample_and_extend(default_means, 10, len(tol_iterations))
projected_default_stds = downsample_and_extend(default_stds, 10, len(tol_iterations))
plot_a_few_trajectories_1d(
    [projected_default_means, projected_default_stds], [tol_means, tol_stds],
    best_mu, best_std, r"$\lambda$", label_prefix="Stan grad100_tol_rel_obj1e-6 ", x=tol_iterations,
    backend=PLOT_BACKEND,
)

projected_default_means_eta = downsample_and_extend(default_means, 10, len(eta_iterations))
projected_default_stds_eta = downsample_and_extend(default_stds, 10, len(eta_iterations))
plot_a_few_trajectories_1d(
    [projected_default_means_eta, projected_default_stds_eta], [eta_means, eta_stds],
    best_mu, best_std, r"$\lambda$", label_prefix="Stan grad100_tol_rel_obj1e-6_eta0.01 ", x=eta_iterations,
    backend=PLOT_BACKEND,
)


In [1]:
import pickle
import sys
from pathlib import Path
import numpy as np

REPO_ROOT = Path("/Users/madelynandersen/Documents/VB Bakeoff/simpleVI")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from modulars.stan_rr_test import stan_result_by_scenario
from modulars.plot_rr import plot_mean_band_rrs_1d
from modulars.utils import load_config, load_best_values, exp_lognormal_moments

NB_DIR = REPO_ROOT / "single_MC/gamma_poisson/random_restarts"

config = load_config(NB_DIR / "../poisson_config.json")
best_mu, best_std = load_best_values(config=config, transform=exp_lognormal_moments, n_samples=50_000, seed=1)

with open(NB_DIR / "stan_cmdstan_output/stan_result.pkl", "rb") as f:
    _main_result = pickle.load(f)
_by_scenario = stan_result_by_scenario(_main_result)
tol_means, tol_stds, tol_iterations = _by_scenario["grad100_tol_rel_obj1e-6"]

with open(NB_DIR / "tol_1e-6_default_rest_result.pkl", "rb") as f:
    _td_result = pickle.load(f)
_td_by_scenario = stan_result_by_scenario(_td_result)
td_means, td_stds, td_iterations = _td_by_scenario["tol_1e-6_default_rest"]

n_td_failures = len(_td_result["failures"])
print(f"tol_1e-6_default_rest: {td_means.shape[0]} restarts, {len(td_iterations)} tracked points, "
      f"final iter={int(td_iterations[-1])}")
print(f"failures: {n_td_failures}")

tol_1e-6_default_rest: 10 restarts, 501 tracked points, final iter=75000
failures: 1002


In [1]:
plot_mean_band_rrs_1d(
    td_means, td_stds, best_mu, best_std,
    td_iterations, r"$\lambda$", "tol_1e-6_default_rest",
    label_prefix="Stan tol_1e-6_default_rest ", backend="plotly",
)

In [1]:
import pickle
import sys
from pathlib import Path
import numpy as np

REPO_ROOT = Path("/Users/madelynandersen/Documents/VB Bakeoff/simpleVI")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from modulars.stan_rr_test import stan_result_by_scenario
from modulars.plot_rr import plot_a_few_trajectories_1d
from modulars.utils import load_config, load_best_values, exp_lognormal_moments

NB_DIR = REPO_ROOT / "single_MC/gamma_poisson/random_restarts"

config = load_config(NB_DIR / "../poisson_config.json")
best_mu, best_std = load_best_values(config=config, transform=exp_lognormal_moments, n_samples=50_000, seed=1)

with open(NB_DIR / "stan_cmdstan_output/stan_result.pkl", "rb") as f:
    _main_result = pickle.load(f)
_by_scenario = stan_result_by_scenario(_main_result)
default_means, default_stds, default_iterations = _by_scenario["default"]

with open(NB_DIR / "tol_1e-6_default_rest_result.pkl", "rb") as f:
    _td_result = pickle.load(f)
_td_by_scenario = stan_result_by_scenario(_td_result)
td_means, td_stds, td_iterations = _td_by_scenario["tol_1e-6_default_rest"]

def downsample_and_extend(arr, factor, target_len):
    downsampled = arr[..., ::factor]
    n_have = downsampled.shape[-1]
    if n_have < target_len:
        pad = np.repeat(downsampled[..., -1:], target_len - n_have, axis=-1)
        downsampled = np.concatenate([downsampled, pad], axis=-1)
    return downsampled

projected_default_means = downsample_and_extend(default_means, 15, len(td_iterations))
projected_default_stds = downsample_and_extend(default_stds, 15, len(td_iterations))
plot_a_few_trajectories_1d(
    [projected_default_means, projected_default_stds], [td_means, td_stds],
    best_mu, best_std, r"$\lambda$", k=2, N=min(default_means.shape[0], td_means.shape[0]),
    label_prefix="Stan tol_1e-6_default_rest ", x=td_iterations, backend="plotly",
)

In [1]:
import pickle
import sys
from pathlib import Path
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

REPO_ROOT = Path("/Users/madelynandersen/Documents/VB Bakeoff/simpleVI")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from modulars.stan_rr_test import stan_result_by_scenario
from modulars.utils import load_config, load_best_values, exp_lognormal_moments

NB_DIR = REPO_ROOT / "single_MC/gamma_poisson/random_restarts"

config = load_config(NB_DIR / "../poisson_config.json")
best_mu, best_std = load_best_values(config=config, transform=exp_lognormal_moments, n_samples=50_000, seed=1)

with open(NB_DIR / "stan_cmdstan_output/stan_result.pkl", "rb") as f:
    _main_result = pickle.load(f)
_by_scenario = stan_result_by_scenario(_main_result)
tol_means, tol_stds, tol_iterations = _by_scenario["grad100_tol_rel_obj1e-6"]

with open(NB_DIR / "tol_1e-6_default_rest_result.pkl", "rb") as f:
    _td_result = pickle.load(f)
_td_by_scenario = stan_result_by_scenario(_td_result)
td_means, td_stds, td_iterations = _td_by_scenario["tol_1e-6_default_rest"]

fig = make_subplots(rows=2, cols=1, vertical_spacing=0.12,
                     subplot_titles=(r"posterior mean of $\lambda$", r"posterior std of $\lambda$"))

fig.add_trace(go.Scatter(x=tol_iterations, y=np.nanmean(tol_means, axis=0), mode="lines",
                          line=dict(color="rgb(148,103,189)", width=2), opacity=0.65,
                          name="grad100_tol_rel_obj1e-6", legendgroup="tol", showlegend=True), row=1, col=1)
fig.add_trace(go.Scatter(x=td_iterations, y=np.nanmean(td_means, axis=0), mode="lines",
                          line=dict(color="rgb(31,119,180)", width=2), opacity=0.65,
                          name="tol_1e-6_default_rest", legendgroup="default_rest", showlegend=True), row=1, col=1)
fig.add_hline(y=best_mu, line=dict(color="red", dash="dash"), row=1, col=1)

fig.add_trace(go.Scatter(x=tol_iterations, y=np.nanmean(tol_stds, axis=0), mode="lines",
                          line=dict(color="rgb(148,103,189)", width=2), opacity=0.65,
                          name="grad100_tol_rel_obj1e-6", legendgroup="tol", showlegend=False), row=2, col=1)
fig.add_trace(go.Scatter(x=td_iterations, y=np.nanmean(td_stds, axis=0), mode="lines",
                          line=dict(color="rgb(31,119,180)", width=2), opacity=0.65,
                          name="tol_1e-6_default_rest", legendgroup="default_rest", showlegend=False), row=2, col=1)
fig.add_hline(y=best_std, line=dict(color="red", dash="dash"), row=2, col=1)
fig.add_trace(go.Scatter(x=[None], y=[None], mode="lines", line=dict(color="red", dash="dash"),
                          name="best variational approximation"), row=1, col=1)

fig.update_xaxes(title_text="iteration", range=[0, 75_000], row=1, col=1)
fig.update_xaxes(title_text="iteration", range=[0, 75_000], row=2, col=1)
fig.update_layout(height=800, width=850, title_text="Stan tol_1e-6_default_rest vs grad100_tol_rel_obj1e-6")
fig.show()

In [4]:
CHECKPOINTS = [50_000, 100_000, 500_000, 1_000_000]

with open(NB_DIR / "default_checkpoint_results.pkl", "rb") as f:
    _cp_results = pickle.load(f)

def pct_error(setup, best):
    return (setup - best) / best * 100.0

def zscore(setup_mean, best_mean, best_std):
    return (setup_mean - best_mean) / best_std

labels = ["lambd"]
best_mean_arr = np.array([best_mu])
best_std_arr = np.array([best_std])

print("===== gam_pois: percent-error / z-score vs best variational approximation =====")
print(f"reference (best) mean: {np.round(best_mean_arr, 4)}")
print(f"reference (best) std : {np.round(best_std_arr, 4)}\n")

rows = []
for cp in CHECKPOINTS:
    r = _cp_results[cp]
    m = np.asarray(r["means_by_scenario"]["default_single"][0, 0, :], dtype=float)
    s = np.asarray(r["stds_by_scenario"]["default_single"][0, 0, :], dtype=float)
    rows.append((f"default, single restart to {cp}", m, s))

for label, scen_name in [
    ("grad100_default_rest (final)", "grad100_default_rest"),
    ("grad100_tol_rel_obj1e-6 (final)", "grad100_tol_rel_obj1e-6"),
    ("grad100_tol_rel_obj1e-6_eta0.01 (final)", "grad100_tol_rel_obj1e-6_eta0.01"),
    ("tol_1e-6_default_rest (final)", "tol_1e-6_default_rest"),
]:
    if scen_name == "tol_1e-6_default_rest":
        means, stds, iters = td_means, td_stds, td_iterations
    else:
        means, stds, iters = _by_scenario[scen_name]
    m = np.array([np.nanmean(means[:, -1])])
    s = np.array([np.nanmean(stds[:, -1])])
    rows.append((f"{label} (iter={int(iters[-1])})", m, s))

for row_name, m, s in rows:
    pe_mean = pct_error(m, best_mean_arr)
    pe_std = pct_error(s, best_std_arr)
    z = zscore(m, best_mean_arr, best_std_arr)
    for j, lab in enumerate(labels):
        print(f"  {row_name:45s} {lab:10s}  pct_err_mean={pe_mean[j]:9.3f}%  pct_err_std={pe_std[j]:9.3f}%  z={z[j]:7.3f}")

===== gam_pois: percent-error / z-score vs best variational approximation =====
reference (best) mean: [1.49]
reference (best) std : [1.4624]

  default, single restart to 50000              lambd       pct_err_mean=   10.349%  pct_err_std=   42.517%  z=  0.105
  default, single restart to 100000             lambd       pct_err_mean=    9.988%  pct_err_std=   25.761%  z=  0.102
  default, single restart to 500000             lambd       pct_err_mean=    3.608%  pct_err_std=   10.058%  z=  0.037
  default, single restart to 1000000            lambd       pct_err_mean=    9.266%  pct_err_std=   24.446%  z=  0.094
  grad100_default_rest (final) (iter=30000)     lambd       pct_err_mean=    0.355%  pct_err_std=    3.483%  z=  0.004
  grad100_tol_rel_obj1e-6 (final) (iter=200000) lambd       pct_err_mean=    0.240%  pct_err_std=    0.052%  z=  0.002
  grad100_tol_rel_obj1e-6_eta0.01 (final) (iter=200000) lambd       pct_err_mean=    1.107%  pct_err_std=   -1.163%  z=  0.011
  tol_1e-6_defau

In [1]:
import pickle
import sys
from pathlib import Path
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

REPO_ROOT = Path("/Users/madelynandersen/Documents/VB Bakeoff/simpleVI")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from modulars.stan_rr_test import stan_result_by_scenario
from modulars.utils import load_config, load_best_values, exp_lognormal_moments

NB_DIR = REPO_ROOT / "single_MC/gamma_poisson/random_restarts"

config = load_config(NB_DIR / "../poisson_config.json")
best_mu, best_std = load_best_values(config=config, transform=exp_lognormal_moments, n_samples=50_000, seed=1)

with open(NB_DIR / "stan_cmdstan_output/stan_result.pkl", "rb") as f:
    _main_result = pickle.load(f)
_by_scenario = stan_result_by_scenario(_main_result)
default_means, default_stds, default_iterations = _by_scenario["default"]
default_trace_mean = default_means[0]
default_trace_std = default_stds[0]

CHECKPOINTS = [50_000, 100_000, 500_000, 1_000_000]
with open(NB_DIR / "default_checkpoint_results.pkl", "rb") as f:
    _cp_results = pickle.load(f)
cp_means = np.array([np.asarray(_cp_results[cp]["means_by_scenario"]["default_single"][0, 0, :], dtype=float)[0] for cp in CHECKPOINTS])
cp_stds = np.array([np.asarray(_cp_results[cp]["stds_by_scenario"]["default_single"][0, 0, :], dtype=float)[0] for cp in CHECKPOINTS])
cp_iters = np.array(CHECKPOINTS)

fig = make_subplots(rows=2, cols=1, vertical_spacing=0.15,
                     subplot_titles=(r"posterior mean of $\lambda$", r"posterior std of $\lambda$"))

fig.add_trace(go.Scatter(x=default_iterations, y=default_trace_mean, mode="lines",
                          line=dict(color="rgb(31,119,180)", width=1.5),
                          name="Single restart (same seed)", legendgroup="line", showlegend=True), row=1, col=1)
fig.add_trace(go.Scatter(x=cp_iters, y=cp_means, mode="markers", marker=dict(color="orange", size=9),
                          name="Single extended run example", legendgroup="dots", showlegend=True), row=1, col=1)
fig.add_hline(y=best_mu, line=dict(color="red", dash="dash"), row=1, col=1)

fig.add_trace(go.Scatter(x=default_iterations, y=default_trace_std, mode="lines",
                          line=dict(color="rgb(31,119,180)", width=1.5),
                          name="Single restart (same seed)", legendgroup="line", showlegend=False), row=2, col=1)
fig.add_trace(go.Scatter(x=cp_iters, y=cp_stds, mode="markers", marker=dict(color="orange", size=9),
                          name="Single extended run example", legendgroup="dots", showlegend=False), row=2, col=1)
fig.add_hline(y=best_std, line=dict(color="red", dash="dash"), row=2, col=1)
fig.add_trace(go.Scatter(x=[None], y=[None], mode="lines", line=dict(color="red", dash="dash"),
                          name="best variational approximation"), row=1, col=1)

fig.update_xaxes(type="log", title_text="iteration (log scale)", row=1, col=1)
fig.update_xaxes(type="log", title_text="iteration (log scale)", row=2, col=1)
fig.update_layout(height=800, width=850, title_text="gam_pois — posterior mean/std, default settings out to long iteration horizons")
fig.show()